# ETL 03 — Extração dos demonstrativos financeiros (CVM/DFP)

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Fonte:** Portal de Dados Abertos da CVM — Demonstrações Financeiras Padronizadas (DFP), arquivos anuais.

**Período:** exercícios fiscais de 2019 a 2023.

**Demonstrativos:** DRE, Balanço Patrimonial Ativo (BPA) e Passivo (BPP), todos consolidados.

**Saída:** CSV filtrado com as 9 companhias da carteira em `data/raw/` (não versionado; reproduzível executando este notebook).

In [1]:
from pathlib import Path
import time
import zipfile
import requests
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
CVM_DIR = RAW_DIR / "cvm"
CVM_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)

Raiz do projeto: c:\Users\waldinei.rosa\Projetos\Projeto_Triade_Financeira_Waldinei


In [ ]:
ANOS = range(2019, 2024)  # 2019 a 2023 (o range não inclui o último número)
URL_DFP = (
    "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
)
DEMONSTRATIVOS = ["DRE", "BPA", "BPP"]  # Resultado, Balanço Ativo e Balanço Passivo

In [ ]:
def baixar_dfp(ano, tentativas=3):
    """Baixa o zip anual da DFP; se o arquivo já existir, reaproveita."""
    destino = CVM_DIR / f"dfp_cia_aberta_{ano}.zip"
    if destino.exists():
        print(
            f"{ano}: já existe ({destino.stat().st_size / 1e6:.1f} MB) — download ignorado"
        )
        return destino
    for tentativa in range(1, tentativas + 1):
        try:
            with requests.get(
                URL_DFP.format(ano=ano), stream=True, timeout=120
            ) as resposta:
                resposta.raise_for_status()
                with open(destino, "wb") as arquivo:
                    for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                        arquivo.write(bloco)
            print(f"{ano}: download concluído ({destino.stat().st_size / 1e6:.1f} MB)")
            return destino
        except requests.RequestException as erro:
            print(f"{ano}: tentativa {tentativa} falhou ({erro})")
            destino.unlink(missing_ok=True)  # remove arquivo incompleto
            time.sleep(3 * tentativa)
    raise RuntimeError(f"Não foi possível baixar a DFP de {ano}.")


arquivos = {}
for ano in ANOS:
    arquivos[ano] = baixar_dfp(ano)
    time.sleep(1)  # pausa entre requisições

2019: download concluído (11.2 MB)
2020: download concluído (12.6 MB)
2021: download concluído (13.3 MB)
2022: download concluído (13.4 MB)
2023: download concluído (13.6 MB)


In [ ]:
def ler_demonstrativo(caminho_zip, ano, demonstrativo):
    """Lê um demonstrativo consolidado de dentro do zip anual, sem precisar extrair."""
    nome = f"dfp_cia_aberta_{demonstrativo}_con_{ano}.csv"
    with zipfile.ZipFile(caminho_zip) as z, z.open(nome) as arquivo:
        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin-1",
            dtype={"CNPJ_CIA": str, "CD_CONTA": str},
        )
    df["DEMONSTRATIVO"] = demonstrativo
    return df

In [ ]:
dre_2023 = ler_demonstrativo(arquivos[2023], 2023, "DRE")
empresas = dre_2023[["CNPJ_CIA", "CD_CVM", "DENOM_CIA"]].drop_duplicates()

padrao = (
    "RENNER|AMBEV|PETROBRAS|ENGIE BRASIL|^VALE S|SUZANO|^WEG S|EMBRAER|UNIBANCO HOLDING"
)
empresas[
    empresas["DENOM_CIA"].str.contains(padrao, case=False, regex=True)
].sort_values("DENOM_CIA")

,CNPJ_CIA,CD_CVM,DENOM_CIA
7432,07.526.557/0001-00,23264,AMBEV S.A.
7736,07.689.002/0001-89,20087,EMBRAER S.A.
3128,02.474.103/0001-19,17329,ENGIE BRASIL ENERGIA S.A.
27326,60.872.504/0001-23,19348,ITAU UNIBANCO HOLDING S.A.
33446,92.754.738/0001-62,8133,LOJAS RENNER S.A.
19968,33.000.167/0001-01,9512,PETROLEO BRASILEIRO S.A. PETROBRAS
26834,60.651.809/0001-05,9067,SUZANO HOLDING S.A.
15170,16.404.287/0001-55,13986,SUZANO S.A.
21176,33.592.510/0001-54,4170,VALE S.A.
31142,84.429.695/0001-11,5410,WEG S.A.


In [ ]:
CNPJ_CARTEIRA = {
    "LREN3": "92.754.738/0001-62",
    "ABEV3": "07.526.557/0001-00",
    "PETR4": "33.000.167/0001-01",
    "EGIE3": "02.474.103/0001-19",
    "VALE3": "33.592.510/0001-54",
    "SUZB3": "16.404.287/0001-55",
    "WEGE3": "84.429.695/0001-11",
    "EMBR3": "07.689.002/0001-89",
    "ITUB4": "60.872.504/0001-23",
}
cnpj_para_ticker = {cnpj: ticker for ticker, cnpj in CNPJ_CARTEIRA.items()}

partes = []
for ano, caminho in arquivos.items():
    for demonstrativo in DEMONSTRATIVOS:
        df = ler_demonstrativo(caminho, ano, demonstrativo)
        df = df[df["CNPJ_CIA"].isin(cnpj_para_ticker)]
        df = df[
            ~df["ORDEM_EXERC"].str.startswith("PEN")
        ]  # mantém só o exercício atual (ÚLTIMO)
        partes.append(df)

dfp = pd.concat(partes, ignore_index=True)

# Reapresentações: mantém apenas a versão mais recente de cada documento
versao_max = dfp.groupby(["CNPJ_CIA", "DT_REFER"])["VERSAO"].transform("max")
dfp = dfp[dfp["VERSAO"] == versao_max]

dfp.insert(0, "TICKER", dfp["CNPJ_CIA"].map(cnpj_para_ticker))
dfp.to_csv(RAW_DIR / "cvm_dfp_carteira_2019_2023.csv", index=False, encoding="utf-8")

print("Total de linhas:", len(dfp))

Total de linhas: 9894


In [7]:
print("Escalas monetárias:", dfp["ESCALA_MOEDA"].unique())
print("Moedas:", dfp["MOEDA"].unique())

dfp.groupby(["TICKER", "DEMONSTRATIVO"])["DT_REFER"].nunique().unstack()

Escalas monetárias: <ArrowStringArray>
['MIL']
Length: 1, dtype: str
Moedas: <ArrowStringArray>
['REAL']
Length: 1, dtype: str


DEMONSTRATIVO,BPA,BPP,DRE
TICKER,,,
ABEV3,5,5,5
EGIE3,5,5,5
EMBR3,5,5,5
ITUB4,5,5,5
LREN3,5,5,5
PETR4,5,5,5
SUZB3,5,5,5
VALE3,5,5,5
WEGE3,5,5,5


## Conferência da extração

| Verificação | Resultado |
|---|---|
| Empresas localizadas por CNPJ | 9 de 9 (CNPJs conferidos com o cadastro da própria base CVM) |
| Exercícios por empresa | 5 (2019 a 2023) em DRE, BPA e BPP |
| Exercício considerado | Apenas o atual de cada arquivo ("ÚLTIMO"); o "PENÚLTIMO" foi descartado para evitar duplicidade |
| Reapresentações | Mantida apenas a versão mais recente de cada documento |
| Moeda / escala | Real (R$), em **milhares** (`ESCALA_MOEDA = MIL`) |

**Observações:**
- A busca por nome retornou também a **Suzano Holding S.A.**, controladora da Suzano. A SUZB3 corresponde à **Suzano S.A.** (CNPJ 16.404.287/0001-55); por isso o filtro é feito por CNPJ, e não por nome.
- Os valores contábeis deverão ser multiplicados por 1.000 (ou rotulados como "R$ mil") nas visualizações.
- O Itaú Unibanco (ITUB4) é uma instituição financeira: sua DRE não segue a estrutura das empresas não financeiras e terá tratamento específico na etapa de modelagem.